# Daily Challenge – Week 7, Day 3: Preprocess & Fine-tune Transformer-based Models
BERT and XLM-RoBERTa on the multilingual NLI dataset (*Contradictory, My Dear Watson*). Run in **Google Colab with a GPU**.

## 1. Understanding BERT and XLM-RoBERTa
- **BERT** (Google, 2018) is a Transformer **encoder** pre-trained on English text with masked language modelling. It reads a sentence **bidirectionally** and produces a contextual vector per token. It uses a **WordPiece** tokenizer with special tokens `[CLS]` (start / sentence summary) and `[SEP]` (separator). Versions include `bert-base-uncased` (12 layers, 110M parameters, lower-cased), `bert-base-cased`, `bert-large` (24 layers, 340M), and `bert-base-multilingual-cased` (104 languages).
- **XLM-RoBERTa** (Facebook AI, 2019) is a **RoBERTa-style** model (more data, no NSP, dynamic masking) trained on **2.5 TB of text in 100 languages**. It uses a **SentencePiece** tokenizer with a 250,000-token vocabulary and the special tokens `<s>` (start), `</s>` (end / separator) and `<pad>`. Versions: `xlm-roberta-base` (270M) and `xlm-roberta-large` (550M). Because this dataset contains **15 languages**, XLM-RoBERTa is the better fit.

In [ ]:
!pip install --quiet transformers sentencepiece datasets

In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
xlmr_tok = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

## 2. Tokenizing Text

In [ ]:
sentence = "The weather is lovely today."
for name, tok in [("BERT", bert_tok), ("XLM-RoBERTa", xlmr_tok)]:
    enc = tok.encode_plus(sentence)
    print(f"--- {name} ---")
    print("Tokens:        ", tok.convert_ids_to_tokens(enc["input_ids"]))
    print("input_ids:     ", enc["input_ids"])
    print("attention_mask:", enc["attention_mask"])
    print("Decoded:       ", tok.decode(enc["input_ids"]))

In [ ]:
# Two-sentence tokenization (premise + hypothesis), as used for NLI
premise, hypothesis = "A man is playing a guitar on stage.", "Someone is performing music."
for name, tok in [("BERT", bert_tok), ("XLM-RoBERTa", xlmr_tok)]:
    enc = tok.encode_plus(premise, hypothesis)
    print(f"--- {name} ---")
    print(tok.convert_ids_to_tokens(enc["input_ids"]))
    if "token_type_ids" in enc:
        print("token_type_ids:", enc["token_type_ids"], "← 0 = first sentence, 1 = second sentence")

BERT marks the two sentences with `[SEP]` plus **token_type_ids** (0/1). XLM-RoBERTa uses `</s></s>` between them and has **no token_type_ids**: it relies on the separators alone.

## 3. Preparing Input Data for the Model

In [ ]:
enc = xlmr_tok.encode_plus(premise, hypothesis, max_length=32, padding="max_length", truncation=True)
print("Tokens:", xlmr_tok.convert_ids_to_tokens(enc["input_ids"]))
print("attention_mask:", enc["attention_mask"])
print("\nSpecial tokens map:", xlmr_tok.special_tokens_map)
print("Vocabulary size – XLM-R:", xlmr_tok.vocab_size, "| BERT:", bert_tok.vocab_size)

- **Padding / truncation:** every input is made exactly `max_length` long: short ones are padded with `<pad>`, long ones are cut, so they can be stacked into batches.
- **Special tokens:** `<s>` starts the sequence (its final vector is used for classification, like BERT's `[CLS]`), and `</s>` ends each sentence.
- **attention_mask:** 1 = real token, 0 = padding. It tells the model to **ignore the padding** in self-attention, so padding doesn't change the result.

## 4. Loading and Exploring the Dataset

In [ ]:
URL = ("https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/"
       "W6D1%20GenAi%20France/Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip")
if not glob.glob("nli_data/**/*.csv", recursive=True):
    urllib.request.urlretrieve(URL, "nli.zip")
    with zipfile.ZipFile("nli.zip") as z:
        z.extractall("nli_data")
csv_files = glob.glob("nli_data/**/*.csv", recursive=True)
print(csv_files)

train = pd.read_csv(next(f for f in csv_files if "train" in os.path.basename(f).lower()))
test = pd.read_csv(next(f for f in csv_files if "test" in os.path.basename(f).lower()))
print("Train:", train.shape, "| Test:", test.shape)
train.head()

In [ ]:
print(train["label"].value_counts().sort_index())       # 0 entailment, 1 neutral, 2 contradiction
print("\nLanguages:\n", train["language"].value_counts())
print("\nColumns needed for training: premise, hypothesis (inputs) and label (target).")

## 5. Creating Cross-Validation Folds

In [ ]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_folds, val_folds = [], []
for fold, (tr_idx, va_idx) in enumerate(kf.split(train, train["label"])):
    train_folds.append(train.iloc[tr_idx].reset_index(drop=True))
    val_folds.append(train.iloc[va_idx].reset_index(drop=True))
    dist = val_folds[-1]["label"].value_counts(normalize=True).sort_index().round(3).tolist()
    print(f"Fold {fold + 1}: train {len(tr_idx)} | val {len(va_idx)} | val label distribution {dist}")

Each fold keeps (almost exactly) the same label proportions as the full dataset, which is what **stratification** guarantees.

## 6. (Bonus) Fine-tuning XLM-RoBERTa on Fold 1
One epoch on one fold, to show the full pipeline (≈ 10 minutes on a Colab GPU). Repeating it for all 5 folds and averaging gives the cross-validated score.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class NLIDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len=96):
        self.enc = tokenizer(list(frame["premise"]), list(frame["hypothesis"]),
                             padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(frame["label"].values)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i], "labels": self.labels[i]}

train_loader = DataLoader(NLIDataset(train_folds[0], xlmr_tok), batch_size=16, shuffle=True)
val_loader = DataLoader(NLIDataset(val_folds[0], xlmr_tok), batch_size=64)

model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS = 1
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * len(train_loader) * EPOCHS), len(train_loader) * EPOCHS)

for epoch in range(EPOCHS):
    model.train()
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        if step % 100 == 0:
            print(f"step {step}/{len(train_loader)} loss {loss.item():.4f}")

model.eval()
correct = total = 0
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.argmax(-1)
        correct += (preds == batch["labels"]).sum().item()
        total += len(preds)
print(f"Fold 1 validation accuracy: {correct / total:.3f}  (majority-class baseline ≈ {val_folds[0]['label'].value_counts(normalize=True).max():.3f})")

**Takeaways:** XLM-RoBERTa handles all 15 languages with a single tokenizer and model, and its accuracy after just one epoch is well above the ~35% majority baseline (typically around **60–70%** with `xlm-roberta-base`; `xlm-roberta-large` and more epochs push it higher). Proper preprocessing (pairs, padding, attention masks) and **stratified k-fold cross-validation** give a reliable estimate of how the model will perform on unseen data.